In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen
import json
import requests
import time
import glob
import os

#helps us with organizing the data
import duckdb

from sklearn.preprocessing import StandardScaler

**How to operate the api:**


*  driver_number: refers the number of the driver. URL of driver number https://en.wikipedia.org/wiki/List_of_Formula_One_driver_numbers#Formula_One_driver_numbers






In [ ]:
session_key = 9468  # one qualifying session
BASE = "https://api.openf1.org/v1"
YEAR = 2023
OUTPUT_DIR = "f1_data_2023"
os.makedirs(OUTPUT_DIR, exist_ok=True) #Prevents crashing if the created file exists

def safe_get(endpoint, params, retries=3):
  """
  Attempts requests.get with retry logic for rate limits and failures.
  """
  for attempt in range(retries):
    try:
      r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=30)
      if r.status_code == 200:
        return r.json()
      elif r.status_code == 429: # limits the rate
         print("Rate Limited, redo in 5s...")
         time.sleep(5)
      else:
        print(f"Status{r.status_code}, retrying...")
        time.sleep(2)
    except Exception as e:
      print(f" Error: {e}, redoing again")
      time.sleep(2)
  print(f"Failed after {retries} attempts: {endpoint} {params}")
  return [] #empty list


#Retrieve aall 2023 qualifying sesion
sessions = safe_get("sessions", {"session_name": "Qualifying", "year": YEAR})
print(f"Found {len(sessions)} qualifying sessions")

pd.DataFrame(sessions).to_parquet(f"{OUTPUT_DIR}/sessions.parquet")

all_drivers = []
all_laps = []
all_stints = []
all_weather = []

for i, sess in enumerate(sessions, 1):
  sk = sess["session_key"]
  country = sess["country_name"]
  print(f"[{i}/{len(sessions)}] Session {sk} — {country} Qualifying")

  # Weather check
  weather = safe_get("weather", {"session_key": sk})
  time.sleep(0.4)
  if any(w.get("rainfall", 0) == 1 for w in weather):
    print(f"Skipping (rainfall detected)")
    continue
  all_weather.extend(weather)

  # driver in this session
  drivers = safe_get("drivers", {"session_key": sk})
  time.sleep(0.4)
  all_drivers.extend(drivers)
  print(f"{len(drivers)} drivers")

  # Laps in this session
  laps = safe_get("laps", {"session_key": sk})
  time.sleep(0.4)
  all_laps.extend(laps)
  print(f"{len(laps)} laps")

  # Stints for this session
  stints = safe_get("stints", {"session_key": sk})
  time.sleep(0.4)
  all_stints.extend(stints)
  print(f"{len(stints)} stints")

  session_telemetry = []
  for d in drivers:
    dn = d["driver_number"]
    samples = safe_get("car_data", {
        "session_key": sk,
        "driver_number": dn
    })
    session_telemetry.extend(samples)
    time.sleep(0.4)

  if session_telemetry:
    df_tel = pd.DataFrame(session_telemetry)
    df_tel.to_parquet(f"{OUTPUT_DIR}/car_data_session_{sk}.parquet")
    print(f"  Saved {len(session_telemetry):,} telemetry samples\n")

print("Saving metadata files")
pd.DataFrame(all_drivers).to_parquet(f"{OUTPUT_DIR}/drivers.parquet")
pd.DataFrame(all_laps).to_parquet(f"{OUTPUT_DIR}/laps.parquet")
pd.DataFrame(all_stints).to_parquet(f"{OUTPUT_DIR}/stints.parquet")
pd.DataFrame(all_weather).to_parquet(f"{OUTPUT_DIR}/weather.parquet")

Found 23 qualifying sessions
[1/23] Session 7768 — Bahrain Qualifying
20 drivers
254 laps
93 stints
  Saved 394,840 telemetry samples

[2/23] Session 7775 — Saudi Arabia Qualifying
20 drivers
329 laps
80 stints
  Saved 360,680 telemetry samples

[3/23] Session 7783 — Australia Qualifying
Skipping (rainfall detected)
[4/23] Session 9064 — Azerbaijan Qualifying
20 drivers
356 laps
101 stints
  Saved 480,740 telemetry samples

[5/23] Session 9074 — United States Qualifying
20 drivers
308 laps
81 stints
  Saved 350,300 telemetry samples

[6/23] Session 9082 — Italy Qualifying
Status404, retrying...
Status404, retrying...
Status404, retrying...
Failed after 3 attempts: weather {'session_key': 9082}
Status404, retrying...
Status404, retrying...
Status404, retrying...
Failed after 3 attempts: drivers {'session_key': 9082}
0 drivers
Status404, retrying...
Status404, retrying...
Status404, retrying...
Failed after 3 attempts: laps {'session_key': 9082}
0 laps
Status404, retrying...
Status404, r

In [ ]:
con = duckdb.connect("f1_2023.duckdb")

con.execute("CREATE OR REPLACE TABLE sessions AS SELECT * FROM 'f1_data_2023/sessions.parquet'")
con.execute("CREATE OR REPLACE TABLE drivers AS SELECT * FROM 'f1_data_2023/drivers.parquet'")
con.execute("CREATE OR REPLACE TABLE laps AS SELECT * FROM 'f1_data_2023/laps.parquet'")
con.execute("CREATE OR REPLACE TABLE stints AS SELECT * FROM 'f1_data_2023/stints.parquet'")
con.execute("CREATE OR REPLACE TABLE weather AS SELECT * FROM 'f1_data_2023/weather.parquet'")
con.execute("CREATE OR REPLACE TABLE car_data AS SELECT * FROM 'f1_data_2023/car_data_session_*.parquet'")

print("Tables created:")
print(con.execute("SHOW TABLES").df())

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tables created:
       name
0  car_data
1   drivers
2      laps
3  sessions
4    stints
5   weather


Checking if the Data was extracted Correctly

In [ ]:
con = duckdb.connect("f1_2023.duckdb")

print("Tables:")
print(con.execute("SHOW TABLES").df())

print("Row Counts")
for tbl in ["sessions", "drivers", "laps", "stints", "weather", "car_data"]:
  n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0] #this fetches data from the first columns of the database
  print(f"{tbl}: {n:,}")

print("Teams:")
print(con.execute("SELECT DISTINCT team_name FROM drivers ORDER BY team_name").df())

print("Drivers by Team")
print(con.execute("""SELECT team_name, COUNT(DISTINCT driver_number) AS n_drivers
FROM drivers
GROUP BY team_name
ORDER BY team_name""").df())

n_sessions = con.execute("""SELECT COUNT(DISTINCT session_key)
FROM car_data""").fetchone()[0]

Tables:
       name
0  car_data
1   drivers
2      laps
3  sessions
4    stints
5   weather
Row Counts
sessions: 23
drivers: 300
laps: 4,754
stints: 1,364
weather: 1,295
car_data: 5,714,560
Teams:
         team_name
0       Alfa Romeo
1       AlphaTauri
2           Alpine
3     Aston Martin
4          Ferrari
5     Haas F1 Team
6          McLaren
7         Mercedes
8  Red Bull Racing
9         Williams
Drivers by Team
         team_name  n_drivers
0       Alfa Romeo          2
1       AlphaTauri          4
2           Alpine          2
3     Aston Martin          2
4          Ferrari          2
5     Haas F1 Team          2
6          McLaren          2
7         Mercedes          2
8  Red Bull Racing          2
9         Williams          2


STEP 1: CLEAN THE DATA

In [ ]:
#--------Lap Times that are useful to use --------

con = duckdb.connect("f1_2023.duckdb")

con.execute("""
  CREATE OR REPLACE TABLE car_data AS
  SELECT * EXCLUDE(date),
    CAST(date AS TIMESTAMP) AS date
  FROM car_data
""")

con.execute("""
  CREATE OR REPLACE TABLE laps AS
  SELECT *
    EXCLUDE(date_start),
    CAST(date_start AS TIMESTAMP) AS date_start
  FROM laps
""")

#Clean-Laps Table

con.execute("""
  CREATE OR REPLACE TABLE clean_laps AS
  WITH session_best AS (
    SELECT
      session_key,
      MIN(lap_duration) AS best_lap
    FROM laps
    WHERE is_pit_out_lap = FALSE
      AND lap_duration IS NOT NULL
      AND lap_duration > 30
    GROUP BY session_key
  )
  SELECT
    l.session_key,
    l.driver_number,
    l.lap_number,
    l.date_start,
    l.lap_duration,
    l.duration_sector_1,
    l.duration_sector_2,
    l.duration_sector_3,
    l.i1_speed,
    l.i2_speed,
    l.st_speed,
    (l.date_start + INTERVAL (l.lap_duration) SECOND) AS date_end
    FROM laps l
    JOIN session_best sb
      ON sb.session_key = l.session_key
    WHERE l.is_pit_out_lap = FALSE
      AND l.lap_duration IS NOT NULL
      AND l.lap_duration <= sb.best_lap * 1.07
      AND l.lap_duration > 30
""")
#Things to --note: 1.07% rule

# Check how many laps survived
n_clean = con.execute("SELECT COUNT(*) FROM clean_laps").fetchone()[0]
print(f"Clean laps: {n_clean:,}")

# Distribution per driver
print(con.execute("""
    SELECT driver_number, COUNT(*) AS n_laps
    FROM clean_laps
    GROUP BY driver_number
    ORDER BY driver_number
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean laps: 1,506
    driver_number  n_laps
0               1      81
1               2      47
2               3      23
3               4      72
4              10      78
5              11      75
6              14      87
7              16      98
8              18      70
9              20      56
10             21      22
11             22      68
12             23      67
13             24      63
14             27      72
15             31      81
16             40      21
17             44      89
18             55      86
19             63      88
20             77      74
21             81      88


This is a part of combining and calculating the features that we will be using to build the model.

In [ ]:
#------------- Compute Features Per Lap ----------------

con = duckdb.connect("f1_2023.duckdb")

con.execute("""
  CREATE OR REPLACE TABLE lap_features AS
  SELECT
    l.session_key,
    l.driver_number,
    l.lap_number,
    l.date_start,
    l.lap_duration,
    l.duration_sector_1,
    l.duration_sector_2,
    l.duration_sector_3,
    l.i1_speed,
    l.i2_speed,
    l.st_speed,

    AVG(c.throttle) AS avg_throttle,
    STDDEV(c.throttle) AS throttle_std,
    AVG(CASE WHEN c.throttle = 100 THEN 1.0 ELSE 0.0 END) AS pct_full_throttle,
    AVG(CASE WHEN c.throttle = 0 THEN 1.0 ELSE 0.0 END) AS pct_zero_throttle,

    AVG(CASE WHEN c.brake = 100 THEN 1.0 ELSE 0.0 END) AS pct_braking,

    AVG(c.n_gear) AS avg_gear,
    COUNT(DISTINCT c.n_gear) AS unique_gears,

    AVG(c.rpm) AS avg_rpm,
    STDDEV(c.rpm) AS rpm_std,
    MAX(c.rpm) AS max_rpm,

    MAX(c.speed) AS max_speed,
    AVG(c.speed) AS avg_speed,
    MIN(c.speed) AS min_speed,
    STDDEV(c.speed) AS speed_std,
    AVG(CASE WHEN c.speed >= 300 THEN 1.0 ELSE 0.0 END) AS pct_above_300,
    AVG(CASE WHEN c.speed >= 200 AND c.speed < 300 THEN 1.0 ELSE 0.0 END) AS pct_200_300,
    AVG(CASE WHEN c.speed >= 100 AND c.speed < 200 THEN 1.0 ELSE 0.0 END) AS pct_100_200,
    AVG(CASE WHEN c.speed < 100 THEN 1.0 ELSE 0.0 END) AS pct_below_100,

    AVG(CASE WHEN c.drs IN (10, 12, 14) THEN 1.0 ELSE 0.0 END) AS pct_drs_active,

    COUNT(*) AS n_samples
  FROM clean_laps AS l
  JOIN car_data AS c
    ON c.session_key = l.session_key
    AND c.driver_number = l.driver_number
    AND c.date >= l.date_start
    AND c.date < l.date_end
  GROUP BY
    l.session_key, l.driver_number, l.lap_number, l.date_start, l.lap_duration,
    l.duration_sector_1, l.duration_sector_2, l.duration_sector_3,
    l.i1_speed, l.i2_speed, l.st_speed
""")

n_features = con.execute("SELECT COUNT(*) FROM lap_features").fetchone()[0]
print(f"Feature rows: {n_features:,}")

con.execute("SELECT * FROM lap_features LIMIT 5").df()

Feature rows: 1,506


,session_key,driver_number,lap_number,date_start,lap_duration,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,max_speed,avg_speed,min_speed,speed_std,pct_above_300,pct_200_300,pct_100_200,pct_below_100,pct_drs_active,n_samples
0,7768,16,5,2023-03-04 15:15:09.670,91.094,29.099,39.134,22.861,242.0,274.0,...,326,213.702006,60,70.944592,0.106017,0.498567,0.306590,0.088825,0.234957,349
1,7768,16,10,2023-03-04 15:38:30.536,91.699,29.301,39.434,22.964,241.0,272.0,...,325,210.605114,58,71.882821,0.088068,0.511364,0.303977,0.096591,0.235795,352
2,7768,16,16,2023-03-04 15:59:35.856,90.000,28.825,38.614,22.561,243.0,273.0,...,325,215.271930,63,71.236785,0.105263,0.526316,0.274854,0.093567,0.219298,342
3,7768,18,2,2023-03-04 15:14:18.031,91.617,29.266,39.428,22.923,241.0,268.0,...,318,211.783237,68,68.925811,0.066474,0.534682,0.309249,0.089595,0.242775,346
4,7768,18,8,2023-03-04 15:37:31.240,92.305,29.581,39.779,22.945,241.0,267.0,...,317,209.099723,63,67.705949,0.066482,0.526316,0.321330,0.085873,0.238227,361


In [ ]:
#GEAR CHANGE Table

con.execute(
    """
    CREATE OR REPLACE TABLE gear_changes AS
    WITH gear_with_lap AS (
      SELECT
        l.session_key,
        l.driver_number,
        l.lap_number,
        c.date,
        c.n_gear,
        LAG(c.n_gear) OVER (
          PARTITION BY l.session_key, l.driver_number, l.lap_number
          ORDER BY c.date
        ) AS prev_gear
        FROM clean_laps AS l
        JOIN car_data c
          ON c.session_key = l.session_key
          AND c.driver_number = l.driver_number
          AND c.date >= l.date_start
          AND c.date < l.date_end
    )
    SELECT
      session_key,
      driver_number,
      lap_number,
      SUM(CASE WHEN n_gear != prev_gear AND prev_gear IS NOT NULL THEN 1 ELSE 0 END) AS n_gear_changes
    FROM gear_with_lap
    GROUP BY session_key, driver_number, lap_number
    """
)

con.execute("""
  CREATE OR REPLACE TABLE lap_features AS
  SELECT
    f.*,
    COALESCE(g.n_gear_changes, 0) AS n_gear_changes
  FROM lap_features f
  LEFT JOIN gear_changes g
  ON g.session_key = f.session_key
  AND g.driver_number = f.driver_number
  AND g.lap_number = f.lap_number
""")

Atatching all Team labels to the data

In [ ]:
con.execute("""
  CREATE OR REPLACE TABLE lap_features_labeled AS
  SELECT
    f.*,
    d.team_name,
    d.full_name AS driver_name
  FROM lap_features AS f
  JOIN drivers d
    ON d.session_key = f.session_key
    AND d.driver_number = f.driver_number
""")

print(con.execute("""
  SELECT team_name, COUNT(*) AS n_laps
  FROM lap_features_labeled
  GROUP BY team_name
  ORDER BY n_laps DESC
""").df())

         team_name  n_laps
0          Ferrari     184
1         Mercedes     177
2          McLaren     160
3           Alpine     159
4     Aston Martin     157
5  Red Bull Racing     156
6       Alfa Romeo     137
7       AlphaTauri     134
8     Haas F1 Team     128
9         Williams     114


In [ ]:
#extract cleaned data for usage

df = con.execute("SELECT * FROM lap_features_labeled").df()

# Save as CSV
df.to_csv("f1_data_2023/lap_features_labeled.csv", index=False)

print(f"Saved {len(df):,} rows to lap_features_labeled.csv")
print(f"Columns: {list(df.columns)}")
df.head()

Saved 1,506 rows to lap_features_labeled.csv
Columns: ['session_key', 'driver_number', 'lap_number', 'date_start', 'lap_duration', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'st_speed', 'avg_throttle', 'throttle_std', 'pct_full_throttle', 'pct_zero_throttle', 'pct_braking', 'avg_gear', 'unique_gears', 'avg_rpm', 'rpm_std', 'max_rpm', 'max_speed', 'avg_speed', 'min_speed', 'speed_std', 'pct_above_300', 'pct_200_300', 'pct_100_200', 'pct_below_100', 'pct_drs_active', 'n_samples', 'n_gear_changes', 'team_name', 'driver_name']


,session_key,driver_number,lap_number,date_start,lap_duration,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,speed_std,pct_above_300,pct_200_300,pct_100_200,pct_below_100,pct_drs_active,n_samples,n_gear_changes,team_name,driver_name
0,7768,16,5,2023-03-04 15:15:09.670,91.094,29.099,39.134,22.861,242.0,274.0,...,70.944592,0.106017,0.498567,0.306590,0.088825,0.234957,349,45.0,Ferrari,Charles LECLERC
1,7768,16,10,2023-03-04 15:38:30.536,91.699,29.301,39.434,22.964,241.0,272.0,...,71.882821,0.088068,0.511364,0.303977,0.096591,0.235795,352,49.0,Ferrari,Charles LECLERC
2,7768,16,16,2023-03-04 15:59:35.856,90.000,28.825,38.614,22.561,243.0,273.0,...,71.236785,0.105263,0.526316,0.274854,0.093567,0.219298,342,47.0,Ferrari,Charles LECLERC
3,7768,18,2,2023-03-04 15:14:18.031,91.617,29.266,39.428,22.923,241.0,268.0,...,68.925811,0.066474,0.534682,0.309249,0.089595,0.242775,346,44.0,Aston Martin,Lance STROLL
4,7768,18,8,2023-03-04 15:37:31.240,92.305,29.581,39.779,22.945,241.0,267.0,...,67.705949,0.066482,0.526316,0.321330,0.085873,0.238227,361,47.0,Aston Martin,Lance STROLL
